# Multiple Linear Regression

Multiple linear regression uses several inputs at once. Each input gets its own weight, and the prediction is the sum. Here we predict **mpg** from the car's weight, horsepower and where it was built.

This notebook goes with the **Multiple Linear Regression** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

**The steps:** load the data → turn the category into 0/1 columns → split → fit → read the weights → check the result.

## 1. Load the data

Each row is a car from the late 1970s and early 1980s. We use two numbers (`weight`, `horsepower`) and one category (`origin`: usa, japan or europe).

The data is the [Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg) data set from the UCI Machine Learning Repository (CC BY 4.0). See `datasets/README.md` for what we changed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/mpg.csv"
dataset = pd.read_csv(DATA_URL)
dataset.head()

## 2. Turn the category into numbers

A model needs numbers, so `origin` becomes one 0/1 column per country. This is called **one-hot encoding** (or dummy variables).

Look at the three columns: every car has exactly one `1`, so the columns always add up to 1. That is the **dummy variable trap**: the three columns together repeat the intercept, and the weights stop being unique. The fix is to **drop one column** with `drop_first=True`. The dropped country (europe) becomes the baseline.

In [ ]:
dataset = dataset.dropna(subset=['horsepower'])   # a few cars have no horsepower listed
all_three = pd.get_dummies(dataset['origin'])
print((all_three.sum(axis=1) == 1).all(), '<- the three columns always add up to 1')

X = pd.get_dummies(dataset[['weight', 'horsepower', 'origin']], drop_first=True).astype(float)
y = dataset['mpg'].values
X.head()

## 3. Split into train and test

We keep 25% of the cars back for the test, so the score tells us how the weights carry over to cars the model has never seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(len(X_train), 'train,', len(X_test), 'test')

## 4. Fit and read the weights

Each weight is the change in predicted mpg for one more unit of that input, with the other inputs held fixed. Weights are only easy to compare when the inputs are on the same scale, so do not read a big weight as a big effect without checking the units.

In [ ]:
regressor = LinearRegression()
regressor.fit(X_train, y_train)
weights = pd.Series(regressor.coef_, index=X.columns)
print(weights.round(4))
print('intercept:', round(regressor.intercept_, 2))

## 5. Check the result

Compare the **test** R² with the one from the simple linear notebook. More inputs usually help, but only the test score can tell you whether they really do.

In [ ]:
y_pred_train = regressor.predict(X_train)
y_pred_test = regressor.predict(X_test)
print('Train R²:', round(r2_score(y_train, y_pred_train), 3))
print('Test R²: ', round(r2_score(y_test, y_pred_test), 3))
print('Test error on average:', round(mean_absolute_error(y_test, y_pred_test), 2), 'mpg')

## Try this

1. Remove `drop_first=True` and fit again. The predictions are almost the same, but look at the weights: they are no longer meaningful.
2. Drop `horsepower` from the inputs. How much does the test R² change?
3. Predict a 3000 lb, 100 hp car from japan. The row must have the same columns as `X`.